# <ins>**Conducting a Recency Frequency Monetary Analysis on Online Retail Data.**</ins>

## <ins> Rationale </ins>

Customer segmentation is a valuable tool for businesses to have as it can identify which customers are the most and least profitable. With this, personalised experiences can be delivered to customers, meeting their needs. As customer satisfaction improves, brand loyalty and revenue for businesses can also increase.

Recency Frequency Monetary (RFM) analysis offers businesses an easily interpreted method of segmenting customers. While there are advantages for the use of clustering techniques such as K-means, clusters produced using this method may not yield actionable categories for customers.

RFM analysis allocates scores for each of the customers for each of its components:
<ul>
<li>Recency: Measuring how recent a customer has visited/made a purchase.</li>
<li>Frequency: How often customers visit/make purchases.</li>
<li>Monetary: How much is spent by the customer.</li>
</ul>

Scores from 1-5 are allocated by splitting the data for each factor into quintiles. For this project the Monetary and Frequency scores will be combined and customer groups will be created from 25 possible scores (1-5 Recency, 1-5 Frequency+Monetary).

This project will use the Online Retail dataset will be used from the UC Irvine Machine Learning Repository. This data contains transactions from an online retail store based in the United Kingdom. All transactions within the data take place 01/12/2010-09/12/2011.

## <ins> Data Collection </ins>

In [2]:
# Imports relevant libraries.
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go

In [3]:
# Fetch online retail data
retail_data = fetch_ucirepo(id=352) 

In [4]:
# Grabs data, lowers column names, drops NA's, changes datatypes of columns, and adds total column.
retail_frame = retail_data["data"].original
retail_frame = retail_frame.rename(columns= lambda x: x.lower())

retail_frame = retail_frame.dropna()
retail_frame.loc[:,"invoicedate"] = pd.to_datetime(retail_frame.invoicedate)
retail_frame.loc[:,"customerid"] = retail_frame.customerid.apply(lambda x: int(x))

retail_frame["total"] = retail_frame["quantity"]*retail_frame["unitprice"]

In [5]:
retail_frame.head(3)

,invoiceno,stockcode,description,quantity,invoicedate,unitprice,customerid,country,total
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00


## <ins> Creating RFM Tables </ins>

### Montary Frame

In [6]:
# Totals are summed for each customer.
m_frame = retail_frame[["customerid","total"]].groupby("customerid")["total"].sum().reset_index()
m_frame = m_frame[m_frame.total > 0].reset_index(drop=True)

In [7]:
m_frame.head(3)

,customerid,total
0,12347.0,4310.00
1,12348.0,1797.24
2,12349.0,1757.55


### Recency + Frequency Frame

In [8]:
# Cancellations are excluded for these tables.
rf_frame = retail_frame[retail_frame.invoiceno.apply(
    lambda x: x[0] != "C")][["customerid","invoicedate"]]

rf_frame["recency"] = rf_frame.invoicedate.max() - rf_frame.invoicedate

rf_frame.recency = pd.to_timedelta(rf_frame.recency)
rf_frame.recency = rf_frame["recency"].dt.days

rf_frame = rf_frame.groupby(["customerid"]).agg({
    "invoicedate":"count",
    "recency":"min"
}).reset_index().rename(columns={"invoicedate":"frequency"})

In [9]:
rf_frame.head(3)

,customerid,frequency,recency
0,12346.0,1,325
1,12347.0,182,1
2,12348.0,31,74


### Joins the 2 Tables

In [10]:
rfm_frame = pd.merge(m_frame,rf_frame,on="customerid",how="right")
rfm_frame = rfm_frame.dropna().reset_index(drop=True)

# Adds quintiles for scoring.
rfm_frame["totfreqSum"] = rfm_frame.total + rfm_frame.frequency
rfm_frame["totfreqQ"] = pd.qcut(rfm_frame.totfreqSum,5,labels=[1,2,3,4,5])
rfm_frame["recQ"] = pd.qcut(rfm_frame.recency,5,labels=[5,4,3,2,1]) 

In [11]:
rfm_frame.head(3)

,customerid,total,frequency,recency,totfreqSum,totfreqQ,recQ
0,12347.0,4310.00,182,1,4492.00,5,5
1,12348.0,1797.24,31,74,1828.24,4,2
2,12349.0,1757.55,73,18,1830.55,4,4


### Assigning groups based on Scores

In [12]:
# Dictionary for mapping scores to groups.
cust_dictionary = {
    5:{1:"New Customers",2:"Potential Loyalists",3:"Potential Loyalists",4:"Champion",5:"Champion"},
    4:{1:"Promising",2:"Potential Loyalists",3:"Potential Loyalists",4:"Loyal Customers",5:"Loyal Customers"},
    3:{1:"About To Sleep",2:"About To Sleep",3:"Needs Attention",4:"Loyal Customers",5:"Loyal Customers"},
    2:{1:"Hibernating",2:"Hibernating",3:"At Risk",4:"At Risk",5:"Can't Lose Them"},
    1:{1:"Hibernating",2:"Hibernating",3:"At Risk",4:"At Risk",5:"Can't Lose Them"}
}

def segment_cust(recencyQ:int,totalfreqQ:int) -> str:
    x = recencyQ
    y = totalfreqQ
    if x>5 or x<1:
        raise ValueError(f"Recency quintile out of range: {x}.")
    elif y>5 or y<1:
        raise ValueError(f"TotalFrequency quintile out of range: {y}.")
    else:
        result = cust_dictionary[x][y]
        return result

In [13]:
# Applies "segment_cust" function to rfm_frame.
seg_frame = rfm_frame.copy()
seg_frame["cust_label"] = seg_frame.apply(lambda x: segment_cust(x.recQ,x.totfreqQ),axis=1)
seg_frame[["totfreqQ","recQ"]] = seg_frame[["totfreqQ","recQ"]].astype(int)

In [14]:
seg_frame.head(3)

,customerid,total,frequency,recency,totfreqSum,totfreqQ,recQ,cust_label
0,12347.0,4310.00,182,1,4492.00,5,5,Champion
1,12348.0,1797.24,31,74,1828.24,4,2,At Risk
2,12349.0,1757.55,73,18,1830.55,4,4,Loyal Customers


In [15]:
# Groups data by assigned scores.
grouped = seg_frame.groupby(["cust_label","totfreqQ","recQ"]).agg({
    "total":"sum",
    "frequency":"sum",
    "recency":"sum",
    "customerid":"count"
}).reset_index()

grouped = grouped.rename(columns={"customerid":"amount"})
grouped = grouped.sort_values(by=["recQ","totfreqQ"],ascending=False).reset_index(drop=True)

In [16]:
grouped.head(3)

,cust_label,totfreqQ,recQ,total,frequency,recency,amount
0,Champion,5,5,3962091.17,147166,1868,400
1,Champion,4,5,294287.92,24158,1090,211
2,Potential Loyalists,3,5,83196.19,7789,707,121


As the data was grouped by assigned scores to ensure the heatmap would draw correctly, however the data across each heatmap segment needed to be the same. So the data was grouped by labels with the totals calculated, this was then merged with the "grouped" dataframe. 

In [17]:
grouped_test = grouped.groupby("cust_label").agg({
    "total":"sum",
    "recency":"sum",
    "frequency":"sum",
    "amount":"sum"
}).reset_index()

grouped = grouped.merge(grouped_test,how="left",left_on="cust_label",right_on="cust_label",suffixes=["_summed(x)","_summed(y)"])

In [18]:
grouped.head(3)

,cust_label,totfreqQ,recQ,total_summed(x),frequency_summed(x),recency_summed(x),amount_summed(x),total_summed(y),recency_summed(y),frequency_summed(y),amount_summed(y)
0,Champion,5,5,3962091.17,147166,1868,400,4256379.09,2958,171324,611
1,Champion,4,5,294287.92,24158,1090,211,4256379.09,2958,171324,611
2,Potential Loyalists,3,5,83196.19,7789,707,121,276740.18,7819,26112,503


In [19]:
# Calculates averages.
grouped["recency_avg"] = grouped["recency_summed(y)"]/grouped["amount_summed(y)"]
grouped["total_avg"] = grouped["total_summed(y)"]/grouped["amount_summed(y)"]
grouped["freq_avg"] = grouped["frequency_summed(y)"]/grouped["amount_summed(y)"]

## <ins> Analysis </ins>

Below is a heatmap of each of the customer segments. Each segment can be hovered over to display:

<ul>
<li>The number of customers in that segment.</li>
<li>Monetary: The average spend for customers in the segment.</li>
<li>Recency: The average number of days since the last visit (09/12/2011 - Most recent visit).</li>
<li>Frequency: The average number of visits for customers in that segment.</li>
<li>Proportion: % of total customers in that segment.</li>
</ul>

In [20]:
extra_cols = grouped[["total_avg","recency_avg","freq_avg"]].round(2)
extra_cols[["recency_avg","freq_avg"]] = extra_cols[["recency_avg","freq_avg"]].round(0)
extra_cols["proportion"] = ((grouped["amount_summed(y)"]/(grouped["amount_summed(x)"].sum()))*100).round(2)

fig = go.Figure(data=go.Heatmap(
    x=grouped.recQ,
    y=grouped.totfreqQ,
    z=grouped["amount_summed(y)"],
    customdata= extra_cols,
    text=grouped.cust_label,
    texttemplate="%{text}",
    hovertemplate="Number in Group: %{z}<br>Average Spend: £%{customdata[0]}<br>Average Days Since Last Visit: %{customdata[1]}<br>Average Number of Transactions During the Year: %{customdata[2]}<br>Proportion of Customers: %{customdata[3]}%<extra></extra>",
    name="",
    colorbar={
        "title":"Amount in Group"
    }
    ))

fig.update_layout(
    title={
        "text":"Customer Segmentation for Online Retail Store Based in the UK (01/12/2010-09/12/2011)",
        "yanchor":"top"
    },
    title_subtitle={
        "text":"Hover over segments for averages."
    },
    yaxis = {
        "title" : {
            "text":"F+M Score (Frequency + Monetary)"
        }
    },
    xaxis = {
        "title" : {
            "text":"R Score (Recency)"
        }
    }
)

fig.show()    

## <ins> Group Evaluation </ins>
### Champions and Loyal Customers
The company's largest group is the Loyal Customers category (781 customers). It is typical for consumers in this group to purchase fairly often and spend a large amount. However, hovering over the heatmap, it can be seen that on average Loyal Customers (£3004.98) spent ~57% less than Champions (£6966.25). <br>
Higher value products could be pushed towards this group, possibly moving them into the Champion category.

### Potential Loyalists
Potential Loyalists on average spent (£550.18) ~82% less than Loyal Customers and purchased less frequently (96 transactions). Other categories of products should be pushed towards this group to encourage more frequent purchases and an increase in spending.

### Can't Lose Them and At Risk
The "Can't Lose Them" category on average had not made a purchase in over 4 months (132 days). However, these customers spent ~26% more (£3790.63) than the Loyal Customers segment. There should be an effort to bring these customers back by listening to their feedback, or suggesting newer products.<br>
Similar strategies can also be used for the "At Risk" group (£935.49 average spend). However, the focus should be on "Can't Lose Them" who on average, brought in more revenue (~305% more), and purchased more frequently (an average difference of 73 transactions).

### Needs Attention and About to Sleep
The "Needs Attention" group's business can be brought back through recommending products they had bought previously. Limited offers may also help this group. The "About to Sleep" group should be recommended popular products so they aren't lost.

### Promising and New Customers
Support could be offered to the customers in the "Promising" and "New Customers" groups such as onboarding in order to further improve the business relationship, as well as build brand awareness/loyalty. 

### Hibernating
Finally, different marketing campaigns could be used to attempt to bring back business from the "Hibernating" group.